## Step1 -  Read video list

### Issue - 调用Google Translator 失败，需优化

In [10]:
import requests
from bs4 import BeautifulSoup
from deep_translator import GoogleTranslator
import urllib3
import time

# 關閉因為 verify=False 而產生的安全警告（InsecureRequestWarning）
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def scrape_ted_ed_series():
    base_url = "https://ed.ted.com/ted_ed_collections"
    # 設定目標系列網址
    series_url = f"{base_url}/getting-under-our-skin"
    
    # 加上 User-Agent 模擬真實瀏覽器
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }

    try:
        print(f"正在連線至系列頁面並處理翻頁: {series_url}")
        
        lesson_links = []
        page = 1
        
        # ==========================================
        # 第 1 階段：自動翻頁抓取所有影片連結
        # ==========================================
        while True:
            page_url = f"{series_url}?page={page}"
            response = requests.get(page_url, headers=headers, verify=False)
            
            # 如果遇到 404 Not Found，代表已經翻到底了，正常結束翻頁
            if response.status_code == 404:
                print(f"第 {page} 頁不存在 (404)，翻頁順利結束。")
                break
                
            # 若為其他伺服器錯誤，則拋出異常
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')

            # 抓取該頁所有的 lessons 連結
            page_links = []
            for a_tag in soup.find_all('a', href=True):
                href = a_tag['href']
                if '/lessons/' in href:
                    # 確保是完整網址
                    full_url = "https://ed.ted.com" + href if href.startswith('/') else href
                    # 清理網址：去除 ? 後面的追蹤參數，避免同一影片被視為不同連結
                    clean_url = full_url.split('?')[0]
                    
                    if clean_url not in lesson_links and clean_url not in page_links:
                        page_links.append(clean_url)

            # 如果這一頁沒有找到任何新的連結，也代表翻頁結束
            if not page_links:
                print(f"第 {page} 頁未找到新影片，翻頁結束。")
                break
                
            lesson_links.extend(page_links)
            print(f"第 {page} 頁找到 {len(page_links)} 個連結，目前累計 {len(lesson_links)} 個。")
            
            page += 1
            time.sleep(0.5) # 稍微暫停，避免頻繁請求被伺服器阻擋

        if not lesson_links:
            print("未找到任何影片連結，請檢查網址或網頁結構。")
            return

        print(f"\n✅ 總共收集到 {len(lesson_links)} 支不重複的影片連結！")
        print("開始逐一進入頁面抓取標題並翻譯...\n")

        # ==========================================
        # 第 2 階段：逐一進入影片頁面抓取標題並翻譯
        # ==========================================
        results = []
        # 設定翻譯器（英翻簡中）
        translator = GoogleTranslator(source='en', target='zh-CN') 

        for idx, lesson_url in enumerate(lesson_links, 1):
            print(f"處理中 ({idx}/{len(lesson_links)}): {lesson_url}")
            
            try:
                # 進入單一 Lesson 頁面
                lesson_resp = requests.get(lesson_url, headers=headers, verify=False)
                lesson_soup = BeautifulSoup(lesson_resp.text, 'html.parser')
                
                # 獲取英文標題
                title_tag = lesson_soup.find('title')
                en_title = title_tag.text.replace(' | TED-Ed', '').replace('\n', '').strip() if title_tag else "Unknown Title"
                
                # 翻譯成簡體中文
                try:
                    zh_title = translator.translate(en_title)
                except Exception:
                    zh_title = "翻譯失敗"
                    
                # 組合字串格式：英文標題-中文標題-TED網頁連結
                formatted_string = f"{en_title} - {zh_title} - {lesson_url}"
                results.append(formatted_string)
                
            except Exception as e:
                print(f"❌ 抓取 {lesson_url} 時發生錯誤: {e}")
                
            # 延遲 0.2 秒，降低伺服器負載與被封鎖的風險
            time.sleep(0.2)

        # ==========================================
        # 第 3 階段：生成 TXT 文檔
        # ==========================================
        output_file = "TED_Ed_Getting_Under_Our_Skin_Links.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            for item in results:
                f.write(item + "\n")
                
        print(f"\n🎉 抓取與翻譯完成！總共 {len(results)} 筆資料已成功匯出至 {output_file}")

    except requests.exceptions.RequestException as e:
        print(f"網路連線錯誤: {e}")
    except Exception as e:
        print(f"發生未預期的錯誤: {e}")

if __name__ == "__main__":
    scrape_ted_ed_series()

正在連線至系列頁面並處理翻頁: https://ed.ted.com/ted_ed_collections/getting-under-our-skin
第 1 頁找到 11 個連結，目前累計 11 個。
第 2 頁找到 11 個連結，目前累計 22 個。
第 3 頁找到 9 個連結，目前累計 31 個。
第 4 頁找到 11 個連結，目前累計 42 個。
第 5 頁找到 12 個連結，目前累計 54 個。
第 6 頁找到 12 個連結，目前累計 66 個。
第 7 頁找到 12 個連結，目前累計 78 個。
第 8 頁找到 11 個連結，目前累計 89 個。
第 9 頁找到 10 個連結，目前累計 99 個。
第 10 頁找到 8 個連結，目前累計 107 個。
第 11 頁找到 11 個連結，目前累計 118 個。
第 12 頁找到 12 個連結，目前累計 130 個。
第 13 頁找到 9 個連結，目前累計 139 個。
第 14 頁找到 7 個連結，目前累計 146 個。
第 15 頁找到 5 個連結，目前累計 151 個。
第 16 頁找到 8 個連結，目前累計 159 個。
第 17 頁找到 10 個連結，目前累計 169 個。
第 18 頁找到 10 個連結，目前累計 179 個。
第 19 頁找到 10 個連結，目前累計 189 個。
第 20 頁找到 10 個連結，目前累計 199 個。
第 21 頁找到 11 個連結，目前累計 210 個。
第 22 頁找到 10 個連結，目前累計 220 個。
第 23 頁找到 11 個連結，目前累計 231 個。
第 24 頁找到 10 個連結，目前累計 241 個。
第 25 頁找到 11 個連結，目前累計 252 個。
第 26 頁找到 11 個連結，目前累計 263 個。
第 27 頁找到 12 個連結，目前累計 275 個。
第 28 頁找到 12 個連結，目前累計 287 個。
第 29 頁找到 7 個連結，目前累計 294 個。
第 30 頁不存在 (404)，翻頁順利結束。

✅ 總共收集到 294 支不重複的影片連結！
開始逐一進入頁面抓取標題並翻譯...

處理中 (1/294): https://ed.ted.com/lessons/how-a-broken-heart-affects-y

## Step2 - Get YT URL for each video

In [12]:
import requests
from bs4 import BeautifulSoup
import re
import urllib3
import time
import os

# 關閉因為 verify=False 而產生的安全警告（InsecureRequestWarning）
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def get_lesson_details(url, headers):
    try:
        response = requests.get(url, headers=headers, verify=False, timeout=10)
        if response.status_code != 200:
            print(f"  [錯誤] 網頁請求失敗，狀態碼: {response.status_code}")
            return "未找到 YouTube 連結 (網頁請求失敗)"
            
        html_text = response.text
        
        # --- 抓取 YouTube 連結 (終極強化版) ---
        youtube_url = "未找到 YouTube 連結"
        
        # 策略 A: 尋找各種變形的 youtubeId, youtube_id, videoId
        yt_id_match = re.search(r'"(?:youtube_?id|videoId|youtubeId)"\s*:\s*"([a-zA-Z0-9_-]{11})"', html_text, re.IGNORECASE)
        
        if yt_id_match:
            video_id = yt_id_match.group(1)
            youtube_url = f"https://www.youtube.com/watch?v={video_id}"
        else:
            # 策略 B: 尋找巢狀結構例如 "youtube": {"id": "xxxxx"}
            nested_match = re.search(r'"youtube"\s*:\s*\{\s*"id"\s*:\s*"([a-zA-Z0-9_-]{11})"', html_text, re.IGNORECASE)
            if nested_match:
                video_id = nested_match.group(1)
                youtube_url = f"https://www.youtube.com/watch?v={video_id}"
            else:
                # 策略 C: 直接在網頁原始碼裡地毯式搜索任何 youtube.com/embed/ 或 watch?v= 的 11 碼 ID
                link_match = re.search(r'youtube\.com/(?:embed/|watch\?v=|v/)([a-zA-Z0-9_-]{11})', html_text)
                if link_match:
                    video_id = link_match.group(1)
                    youtube_url = f"https://www.youtube.com/watch?v={video_id}"

        return youtube_url

    except Exception as e:
        print(f"  [異常] 請求過程中發生錯誤: {e}")
        return "未找到 YouTube 連結 (請求異常)"

def process_links_file():
    input_file = "TED_Ed_Getting_Under_Our_Skin_Links.txt"
    output_file = "TED_Ed_YouTube_Links_Final.txt"
    
    # 檢查輸入檔案是否存在
    if not os.path.exists(input_file):
        print(f"找不到檔案 {input_file}，請確認第一步是否已成功執行並產生此檔案。")
        return

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }

    final_results = []
    
    print(f"開始讀取檔案: {input_file}")
    
    # 開啟檔案並逐行讀取
    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
    total_lines = len(lines)
    print(f"共找到 {total_lines} 筆資料，準備開始抓取 YouTube 連結...\n")

    for idx, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            continue
            
        # 以 " - " 切割字串，假設格式為: "英文標題 - 中文標題 - TED網址"
        # 使用 maxsplit=2，確保即使標題中含有 " - " 也不會被錯誤切斷
        parts = line.rsplit(" - ", 2)
        
        if len(parts) >= 3:
            en_title = parts[0]
            zh_title = parts[1]
            ted_url = parts[2]
            
            print(f"進度 ({idx}/{total_lines}) 正在解析: {ted_url}")
            
            # 呼叫函式抓取 YouTube 連結
            yt_url = get_lesson_details(ted_url, headers)
            
            # 組合最終字串
            final_string = f"{en_title} - {zh_title} - {yt_url}"
            final_results.append(final_string)
            print(f"  => 取得 YT 連結: {yt_url}")
            
        else:
            print(f"  [警告] 忽略格式不符的資料: {line}")
            final_results.append(line)
            
        # 加入微小延遲，避免短時間內發送過多請求被伺服器封鎖
        time.sleep(0.3)

    # 將結果寫入新的檔案
    with open(output_file, "w", encoding="utf-8") as f:
        for item in final_results:
            f.write(item + "\n")
            
    print(f"\n🎉 處理完成！已將 {len(final_results)} 筆資料匯出至 {output_file}")

if __name__ == "__main__":
    process_links_file()

開始讀取檔案: TED_Ed_Getting_Under_Our_Skin_Links.txt
共找到 294 筆資料，準備開始抓取 YouTube 連結...

進度 (1/294) 正在解析: https://ed.ted.com/lessons/how-a-broken-heart-affects-your-body-roni-shanoada
  => 取得 YT 連結: https://www.youtube.com/watch?v=DbEZmLoGDbk
進度 (2/294) 正在解析: https://ed.ted.com/lessons/why-is-getting-bitten-by-a-rabid-animal-so-dangerous-charles-rupprecht
  => 取得 YT 連結: https://www.youtube.com/watch?v=gePOX4nbMU4
進度 (3/294) 正在解析: https://ed.ted.com/lessons/rnai-slicing-dicing-and-serving-your-cells-alex-dainis
  => 取得 YT 連結: https://www.youtube.com/watch?v=tzlGU5EI9rU
進度 (4/294) 正在解析: https://ed.ted.com/lessons/why-do-we-have-to-wear-sunscreen-kevin-p-boyd
  => 取得 YT 連結: https://www.youtube.com/watch?v=ZSJITdsTze0
進度 (5/294) 正在解析: https://ed.ted.com/lessons/a-new-way-to-diagnose-autism-ami-klin
  => 取得 YT 連結: https://www.youtube.com/watch?v=b-J8d1zfRIM
進度 (6/294) 正在解析: https://ed.ted.com/lessons/what-is-fat-george-zaidan
  => 取得 YT 連結: https://www.youtube.com/watch?v=QhUrc4BnPgg
進度 (7/294) 正在

## 临时修复重新翻译标题

In [2]:
import time
import os
import httpx
from googletrans import Translator

# ==========================================
# 【關鍵修復】攔截 googletrans 底層的 httpx 連線，強制關閉 SSL 驗證
# ==========================================
original_client_init = httpx.Client.__init__

def custom_client_init(self, *args, **kwargs):
    kwargs['verify'] = False  # 強制關閉 SSL 驗證
    original_client_init(self, *args, **kwargs)

httpx.Client.__init__ = custom_client_init
# ==========================================

def fix_failed_translations():
    # 設定輸入與輸出的檔案名稱
    input_file = "TED_Ed_YouTube_Links_Final.txt"
    output_file = "TED_Ed_YouTube_Links_Fixed.txt"

    if not os.path.exists(input_file):
        print(f"找不到檔案 {input_file}，請確認檔案名稱與路徑是否正確。")
        return

    # 讀取檔案內容
    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    translator = Translator()
    fixed_results = []
    failed_count = 0
    fixed_count = 0

    print(f"開始掃描 {input_file}，尋找需要重新翻譯的標題...\n")

    for idx, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            continue

        # 從右邊切兩刀，完美分離出: 英文標題、中文標題、網址
        parts = line.rsplit(" - ", 2)
        
        if len(parts) >= 3:
            en_title = parts[0]
            zh_title = parts[1]
            yt_url = parts[2]

            # 檢查中文標題欄位是否為翻譯失敗
            if zh_title in ["翻譯失敗", "翻译失败", "暂无翻译"]:
                failed_count += 1
                print(f"[{idx}/{len(lines)}] 發現缺失翻譯: {en_title}")
                
                new_zh_title = zh_title
                max_retries = 3
                
                # 重新嘗試翻譯
                for attempt in range(max_retries):
                    try:
                        translation = translator.translate(en_title, src='en', dest='zh-cn')
                        new_zh_title = translation.text
                        print(f"  => 修復成功: {new_zh_title}")
                        fixed_count += 1
                        break
                    except Exception as e:
                        print(f"  [重試 {attempt+1}/{max_retries}] 翻譯連線錯誤等待中... ({e})")
                        time.sleep(2)
                        
                # 將修復後的新標題重新組合
                fixed_string = f"{en_title} - {new_zh_title} - {yt_url}"
                fixed_results.append(fixed_string)
                
                # 延遲 1 秒避免再次被 Google 擋下
                time.sleep(1)
            else:
                # 若原本就翻譯成功，則直接保留原樣
                fixed_results.append(line)
        else:
            # 格式不符的行（例如空行或錯誤數據），直接保留
            fixed_results.append(line)

    # 將結果寫入新的 txt 檔案
    with open(output_file, "w", encoding="utf-8") as f:
        for item in fixed_results:
            f.write(item + "\n")

    print("\n" + "="*40)
    print(f"🎉 修復作業完成！")
    print(f"共發現 {failed_count} 筆翻譯失敗，成功修復了 {fixed_count} 筆。")
    print(f"新的完整清單已儲存至: {output_file}")
    print("="*40)

if __name__ == "__main__":
    fix_failed_translations()

開始掃描 TED_Ed_YouTube_Links_Final.txt，尋找需要重新翻譯的標題...

[1/294] 發現缺失翻譯: How a broken heart affects your body
  => 修復成功: 破碎的心如何影响你的身体
[2/294] 發現缺失翻譯: Why is getting bitten by a rabid animal so dangerous? -
  => 修復成功: 为什么被患有狂犬病的动物咬伤如此危险？-
[3/294] 發現缺失翻譯: RNAi: Slicing, dicing and serving your cells - Alex Dainis
  => 修復成功: RNAi：对细胞进行切片、切块和服务 - Alex Dainis
[4/294] 發現缺失翻譯: Why do we have to wear sunscreen? - Kevin P. Boyd
  => 修復成功: 为什么我们必须涂防晒霜？- 凯文·P·博伊德
[5/294] 發現缺失翻譯: A new way to diagnose autism - Ami Klin
  => 修復成功: 诊断自闭症的新方法 - Ami Klin
[6/294] 發現缺失翻譯: What is fat? - George Zaidan
  => 修復成功: 什么是脂肪？——乔治·扎伊丹
[7/294] 發現缺失翻譯: A universal translator for surgeons - Steven Schwaitzberg
  => 修復成功: 外科医生的通用翻译器 - Steven Schwaitzberg
[8/294] 發現缺失翻譯: The mystery of chronic pain - Elliot Krane
  => 修復成功: 慢性疼痛之谜 - Elliot Krane
[9/294] 發現缺失翻譯: Printing a human kidney - Anthony Atala
  => 修復成功: 打印人类肾脏 - 安东尼·阿塔拉
[10/294] 發現缺失翻譯: HIV and flu -- the vaccine strategy - Seth Berkley
  => 修復成功: HIV 和流感——疫苗策略 - 

## Step3: Get subtitle and translate to CN without timeline - txt

In [ ]:
import os
import re
import time
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter

def sanitize_filename(filename):
    """清理檔案名稱中不合法的字元，避免存檔失敗"""
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def extract_video_id(youtube_url):
    """從 YouTube 網址中提取 11 碼的影片 ID"""
    match = re.search(r'(?:v=|/)([a-zA-Z0-9_-]{11})', youtube_url)
    return match.group(1) if match else None

def download_chinese_subtitles():
    input_file = "TED_Ed_YouTube_Links_Fixed.txt"
    # 建立一個資料夾來存放所有下載的字幕
    output_dir = "TED_Ed_Subtitles"
    
    if not os.path.exists(input_file):
        print(f"找不到檔案 {input_file}，請確認第二步是否已完成。")
        return
        
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"開始讀取 {input_file}，準備下載字幕...")

    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    total_lines = len(lines)
    formatter = TextFormatter() # 用來將字幕轉為純文字 (去掉時間軸)

    for idx, line in enumerate(lines, 1):
        line = line.strip()
        if not line or "未找到 YouTube 連結" in line:
            continue
            
        parts = line.split(" - ", 2)
        if len(parts) >= 3:
            en_title = parts[0]
            zh_title = parts[1]
            yt_url = parts[2]
            
            video_id = extract_video_id(yt_url)
            if not video_id:
                print(f"[{idx}/{total_lines}] ❌ 無法解析 Video ID: {yt_url}")
                continue
                
            print(f"[{idx}/{total_lines}] 正在處理: {en_title}")
            
            try:
                # 獲取該影片的字幕列表
                transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
                
                # 策略 1: 嘗試尋找原生的簡體/繁體中文字幕
                try:
                    transcript = transcript_list.find_transcript(['zh-Hans', 'zh-CN', 'zh-TW', 'zh-Hant', 'zh'])
                    fetched_transcript = transcript.fetch()
                    print("  => 找到原生中文字幕！")
                    
                # 策略 2: 如果沒有中文字幕，則抓取英文字幕 (或自動生成的英文字幕) 並翻譯成簡體中文
                except Exception:
                    print("  => 無原生中文字幕，改抓取英文並即時翻譯為簡體中文...")
                    # 尋找英文，然後翻譯成簡體中文 ('zh-Hans')
                    transcript = transcript_list.find_transcript(['en'])
                    translated_transcript = transcript.translate('zh-Hans')
                    fetched_transcript = translated_transcript.fetch()

                # 將字幕格式化為純文字 (TXT) 格式，不要時間軸
                text_data = formatter.format_transcript(fetched_transcript)
                
                # 組合安全的檔名：例如 "1_How does asthma work.txt"
                safe_title = sanitize_filename(en_title)
                filename = os.path.join(output_dir, f"{idx:03d}_{safe_title}.txt")
                
                # 寫入純文字檔
                with open(filename, "w", encoding="utf-8") as txt_file:
                    txt_file.write(f"英文標題: {en_title}\n")
                    txt_file.write(f"中文標題: {zh_title}\n")
                    txt_file.write(f"影片連結: {yt_url}\n")
                    txt_file.write("-" * 40 + "\n\n")
                    txt_file.write(text_data)
                    
                print(f"  ✅ 成功儲存字幕: {filename}")

            except Exception as e:
                print(f"  ❌ 抓取字幕失敗: {e}")
                
            # 保護機制，避免 API 請求過度頻繁
            time.sleep(1)

    print("\n🎉 所有字幕下載作業完成！請查看 TED_Ed_Subtitles 資料夾。")

if __name__ == "__main__":
    download_chinese_subtitles()

## Step3: Get subtitle and translate to CN with timeline - src

In [1]:
import os
import re
import time
from youtube_transcript_api import YouTubeTranscriptApi
# 【關鍵修改 1】引入 SRT 格式化工具
from youtube_transcript_api.formatters import SRTFormatter

def sanitize_filename(filename):
    """清理檔案名稱中不合法的字元，避免存檔失敗"""
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def extract_video_id(youtube_url):
    """從 YouTube 網址中提取 11 碼的影片 ID"""
    match = re.search(r'(?:v=|/)([a-zA-Z0-9_-]{11})', youtube_url)
    return match.group(1) if match else None

def download_chinese_subtitles():
    # 建議讀取經過 AI 翻譯標題修復過的檔案
    input_file = "TED_Ed_YouTube_Links_Fixed.txt"
    # 建立一個新的資料夾來存放 SRT 字幕
    output_dir = "TED_Ed_Subtitles_SRT"
    
    if not os.path.exists(input_file):
        print(f"找不到檔案 {input_file}，請確認前一步驟是否已完成。")
        return
        
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"開始讀取 {input_file}，準備下載 SRT 格式字幕...")

    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    total_lines = len(lines)
    # 【關鍵修改 2】使用 SRTFormatter 取代 TextFormatter
    formatter = SRTFormatter()

    for idx, line in enumerate(lines, 1):
        line = line.strip()
        if not line or "未找到 YouTube 連結" in line:
            continue
            
        parts = line.rsplit(" - ", 2)
        if len(parts) >= 3:
            en_title = parts[0]
            zh_title = parts[1]
            yt_url = parts[2]
            
            video_id = extract_video_id(yt_url)
            if not video_id:
                print(f"[{idx}/{total_lines}] ❌ 無法解析 Video ID: {yt_url}")
                continue
                
            print(f"[{idx}/{total_lines}] 正在處理: {en_title}")
            
            try:
                # 獲取該影片的字幕列表
                transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
                
                # 策略 1: 嘗試尋找原生的簡體/繁體中文字幕
                try:
                    transcript = transcript_list.find_transcript(['zh-Hans', 'zh-CN', 'zh-TW', 'zh-Hant', 'zh'])
                    fetched_transcript = transcript.fetch()
                    print("  => 找到原生中文字幕！")
                    
                # 策略 2: 如果沒有中文字幕，則抓取英文並翻譯成簡體中文
                except Exception:
                    print("  => 無原生中文字幕，改抓取英文並即時翻譯為簡體中文...")
                    transcript = transcript_list.find_transcript(['en'])
                    translated_transcript = transcript.translate('zh-Hans')
                    fetched_transcript = translated_transcript.fetch()

                # 【關鍵修改 3】將字幕資料轉換為標準的 SRT 格式字串
                srt_data = formatter.format_transcript(fetched_transcript)
                
                # 組合安全的檔名，並【將副檔名改為 .srt】
                safe_title = sanitize_filename(zh_title) # 這裡改用中文標題當檔名，方便你在剪輯時辨識
                filename = os.path.join(output_dir, f"{idx:03d}_{safe_title}.srt")
                
                # 寫入檔案 (注意：SRT 檔案不需要額外寫入標題和連結，否則會破壞 SRT 的標準格式)
                with open(filename, "w", encoding="utf-8") as srt_file:
                    srt_file.write(srt_data)
                    
                print(f"  ✅ 成功儲存字幕: {filename}")

            except Exception as e:
                print(f"  ❌ 抓取字幕失敗: {e}")
                
            # 保護機制，避免 API 請求過度頻繁
            time.sleep(1)

    print("\n🎉 所有 SRT 字幕下載作業完成！請查看 TED_Ed_Subtitles_SRT 資料夾。")

if __name__ == "__main__":
    download_chinese_subtitles()

開始讀取 TED_Ed_YouTube_Links_Fixed.txt，準備下載 SRT 格式字幕...
[1/294] 正在處理: How a broken heart affects your body
  ❌ 抓取字幕失敗: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'
[2/294] 正在處理: Why is getting bitten by a rabid animal so dangerous? -
  ❌ 抓取字幕失敗: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'
[3/294] 正在處理: RNAi: Slicing, dicing and serving your cells - Alex Dainis - RNAi：对细胞进行切片、切块和服务
  ❌ 抓取字幕失敗: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'
[4/294] 正在處理: Why do we have to wear sunscreen? - Kevin P. Boyd
  ❌ 抓取字幕失敗: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'


KeyboardInterrupt: 

In [2]:
import os
import re
import time
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter

def sanitize_filename(filename):
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def extract_video_id(youtube_url):
    match = re.search(r'(?:v=|/)([a-zA-Z0-9_-]{11})', youtube_url)
    return match.group(1) if match else None

def download_chinese_subtitles():
    input_file = "TED_Ed_YouTube_Links_Fixed.txt" # 确保这里是你的输入文件名
    output_dir = "TED_Ed_Subtitles_SRT"
    
    if not os.path.exists(input_file):
        print(f"找不到档案 {input_file}。")
        return
        
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    formatter = SRTFormatter()

    for idx, line in enumerate(lines, 1):
        line = line.strip()
        if not line or "未找到 YouTube" in line:
            continue
            
        parts = line.rsplit(" - ", 2)
        if len(parts) >= 3:
            en_title, zh_title, yt_url = parts[0], parts[1], parts[2]
            
            video_id = extract_video_id(yt_url)
            if not video_id:
                continue
                
            print(f"[{idx}/{len(lines)}] 正在处理: {en_title}")
            
            try:
                # 【直接获取字幕】：按照优先级，先找简体，再找繁体，再找中文，最后保底拿英文
                fetched_transcript = YouTubeTranscriptApi.get_transcript(
                    video_id, 
                    languages=['zh-Hans', 'zh-CN', 'zh-TW', 'zh-Hant', 'zh', 'en']
                )
                
                # 转成 SRT 格式
                srt_data = formatter.format_transcript(fetched_transcript)
                
                # 存盘
                safe_title = sanitize_filename(zh_title)
                filename = os.path.join(output_dir, f"{idx:03d}_{safe_title}.srt")
                
                with open(filename, "w", encoding="utf-8") as srt_file:
                    srt_file.write(srt_data)
                    
                print(f"  ✅ 成功储存 SRT 字幕: {filename}")

            except Exception as e:
                print(f"  ❌ 抓取失败: {e}")
                
            time.sleep(1)

if __name__ == "__main__":
    download_chinese_subtitles()

[1/294] 正在处理: How a broken heart affects your body
  ❌ 抓取失败: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'
[2/294] 正在处理: Why is getting bitten by a rabid animal so dangerous? -
  ❌ 抓取失败: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'
[3/294] 正在处理: RNAi: Slicing, dicing and serving your cells - Alex Dainis - RNAi：对细胞进行切片、切块和服务
  ❌ 抓取失败: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'
[4/294] 正在处理: Why do we have to wear sunscreen? - Kevin P. Boyd
  ❌ 抓取失败: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'
[5/294] 正在处理: A new way to diagnose autism - Ami Klin - 诊断自闭症的新方法
  ❌ 抓取失败: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'
[6/294] 正在处理: What is fat? - George Zaidan
  ❌ 抓取失败: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'
[7/294] 正在处理: A universal translator for surgeons - Steven Schwaitzberg - 外科医生的通用翻译器
  ❌ 抓取失败: type object 'YouTubeTranscriptApi' has no a


KeyboardInterrupt



In [4]:
import os
import yt_dlp
import time

def download_test_video_and_subtitles():
    input_file = "TED_Ed_YouTube_Links_Fixed.txt" 
    output_dir = "TED_Ed_Subtitles_SRT"
    
    if not os.path.exists(input_file):
        print(f"找不到檔案 {input_file}，請確認檔名是否正確。")
        return
        
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"開始讀取 {input_file}，啟動 yt-dlp 測試下載引擎...")

    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    total_lines = len(lines)

    for idx, line in enumerate(lines, 1):
        line = line.strip()
        if not line or "未找到 YouTube" in line:
            continue
            
        parts = line.rsplit(" - ", 2)
        if len(parts) >= 3:
            en_title = parts[0]
            zh_title = parts[1]
            yt_url = parts[2]
            
            print(f"\n[{idx}/{total_lines}] 正在測試下載: {en_title}")
            
            # 清理檔名中的不合法字元
            safe_title = "".join(c for c in zh_title if c not in r'\/*?:"<>|')
            
            ydl_opts = {
                'skip_download': False,      # 設為 False 才會同時下載影片
                'format': 'bestvideo+bestaudio/best', # 下載最佳影音並自動合併
                'writesubtitles': True,      # 下載人工上傳字幕
                'writeautomaticsub': True,   # 下載自動生成字幕
                'subtitleslangs': [ 'zh-CN', 'zh-TW', 'en'], #'zh-Hans','zh', 
                'subtitlesformat': 'srt',
                'outtmpl': os.path.join(output_dir, f"{idx:03d}_{safe_title}.%(ext)s"),
                'sleep_interval': 2,
                'max_sleep_interval': 4,
                'extractor_args': {
                    'youtube': {
                        'player_client': ['android', 'web']
                    }
                }
            }
            
            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([yt_url])
                print(f"  ✅ 測試下載成功！")
            except Exception as e:
                print(f"  ❌ 測試下載失敗: {e}")
            
            # 【關鍵修改】下載完第一筆後直接跳出迴圈，實現「只下載一個測試」
            print("\n🔍 測試完成！已自動停止，請檢查資料夾中的影片與字幕檔案。")
            break

if __name__ == "__main__":
    download_test_video_and_subtitles()

開始讀取 TED_Ed_YouTube_Links_Fixed.txt，啟動 yt-dlp 測試下載引擎...

[1/294] 正在測試下載: How a broken heart affects your body
[youtube] Extracting URL: https://www.youtube.com/watch?v=DbEZmLoGDbk
[youtube] DbEZmLoGDbk: Downloading webpage
[youtube] DbEZmLoGDbk: Downloading android player API JSON
[youtube] DbEZmLoGDbk: Downloading web client config
[youtube] DbEZmLoGDbk: Downloading web player API JSON


[info] DbEZmLoGDbk: Downloading subtitles: zh-Hans, zh-CN, zh-TW, en
[info] DbEZmLoGDbk: Downloading 1 format(s): 18
[info] Writing video subtitles to: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-Hans.srt


[download] Destination: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-Hans.srt
[download] 100% of    8.72KiB in 00:00:01 at 4.46KiB/s
[info] Writing video subtitles to: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-CN.srt


[download] Destination: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-CN.srt
[download] 100% of    6.48KiB in 00:00:00 at 33.12KiB/s
[info] Writing video subtitles to: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-TW.srt


[download] Destination: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-TW.srt
[download] 100% of    6.39KiB in 00:00:00 at 33.93KiB/s
[info] Writing video subtitles to: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.en.srt


[download] Destination: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.en.srt
[download] 100% of    6.61KiB in 00:00:00 at 31.83KiB/s
[download] Sleeping 2.62 seconds ...
[download] Destination: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.mp4
[download] 100% of   18.24MiB in 00:00:07 at 2.53MiB/s     
  ✅ 測試下載成功！

🔍 測試完成！已自動停止，請檢查資料夾中的影片與字幕檔案。


In [6]:
import os
import yt_dlp
import re
import time

def fix_srt_case(file_path):
    """
    1. 將 SRT 字幕中的全大寫英文轉為正常大小寫
    2. 取消自動分行，合併為完整的一句
    3. 將全形中文標點符號（如 ’ ‘ “ ”）替換為標準英文半形標點
    """
    if not os.path.exists(file_path):
        return
        
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
        
    blocks = content.strip().split('\n\n')
    new_blocks = []
    
    for block in blocks:
        lines = block.split('\n')
        if len(lines) >= 3:
            index_line = lines[0]
            time_line = lines[1]
            text_lines = lines[2:]
            
            # 將該時間區段的所有文字行合併成完整的一句
            combined_text = " ".join([line.strip() for line in text_lines])
            
            # 【新增】將全形/中文特有引號或標點替換為標準半形英文標點
            # 替換單引號
            combined_text = combined_text.replace('‘', "'").replace('’', "'")
            # 替換雙引號
            combined_text = combined_text.replace('“', '"').replace('”', '"')
            # 替換其他可能混入的全形符號（例如全形逗號、句號）
            combined_text = combined_text.replace('，', ', ').replace('．', '. ')
            
            # 清理因替換產生的多餘空格
            combined_text = re.sub(r'\s+', ' ', combined_text).strip()
            
            # 針對英文：如果整句都是大寫，則轉為正常大小寫
            if combined_text.isupper():
                combined_text = combined_text.capitalize()
                
            # 重新組裝成標準的 SRT 區塊（只有一行文字）
            new_block = f"{index_line}\n{time_line}\n{combined_text}"
            new_blocks.append(new_block)
        else:
            new_blocks.append(block)
            
    # 寫回檔案
    with open(file_path, "w", encoding="utf-8") as f:
        f.write('\n\n'.join(new_blocks) + '\n')
        
    print(f"  ✨ 已自動修正大小寫、取消分行並修復標點符號: {os.path.basename(file_path)}")

def download_test_video_and_subtitles():
    input_file = "TED_Ed_YouTube_Links_Fixed.txt" 
    output_dir = "TED_Ed_Subtitles_SRT"
    
    if not os.path.exists(input_file):
        print(f"找不到檔案 {input_file}，請確認檔名是否正確。")
        return
        
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"開始讀取 {input_file}，啟動 yt-dlp 測試下載引擎...")

    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    total_lines = len(lines)

    for idx, line in enumerate(lines, 1):
        line = line.strip()
        if not line or "未找到 YouTube" in line:
            continue
            
        parts = line.rsplit(" - ", 2)
        if len(parts) >= 3:
            en_title = parts[0]
            zh_title = parts[1]
            yt_url = parts[2]
            
            print(f"\n[{idx}/{total_lines}] 正在測試下載: {en_title}")
            
            # 清理檔名中的不合法字元
            safe_title = "".join(c for c in zh_title if c not in r'\/*?:"<>|')
            
            ydl_opts = {
                'skip_download': False,      # 同時下載影片與字幕
                'format': 'bestvideo+bestaudio/best',
                'writesubtitles': True,      
                'writeautomaticsub': True,  
                'writethumbnail': True,  #封面图
                'subtitleslangs': ['zh-CN', 'zh-TW', 'en'], 
                'subtitlesformat': 'srt',
                'outtmpl': os.path.join(output_dir, f"{idx:03d}_{safe_title}.%(ext)s"),
                'sleep_interval': 2,
                'max_sleep_interval': 4,
                'extractor_args': {
                    'youtube': {
                        'player_client': ['android', 'web']
                    }
                }
            }
            
            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([yt_url])
                print(f"  ✅ 測試下載成功！")
                
                # 下載完成後，自動尋找該字幕檔並進行清理
                for file in os.listdir(output_dir):
                    if file.startswith(f"{idx:03d}") and file.endswith(".srt"):
                        srt_path = os.path.join(output_dir, file)
                        fix_srt_case(srt_path)
                        
            except Exception as e:
                print(f"  ❌ 測試下載失敗: {e}")
            
            print("\n🔍 測試完成！已自動停止，請檢查資料夾中的影片與字幕檔案。")
            break

if __name__ == "__main__":
    download_test_video_and_subtitles()

開始讀取 TED_Ed_YouTube_Links_Fixed.txt，啟動 yt-dlp 測試下載引擎...

[1/294] 正在測試下載: How a broken heart affects your body
[youtube] Extracting URL: https://www.youtube.com/watch?v=DbEZmLoGDbk
[youtube] DbEZmLoGDbk: Downloading webpage
[youtube] DbEZmLoGDbk: Downloading android player API JSON
[youtube] DbEZmLoGDbk: Downloading web client config
[youtube] DbEZmLoGDbk: Downloading web player API JSON


[info] DbEZmLoGDbk: Downloading subtitles: zh-CN, zh-TW, en
[info] DbEZmLoGDbk: Downloading 1 format(s): 18
Deleting existing file TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-CN.srt
[info] Writing video subtitles to: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-CN.srt


[download] Destination: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-CN.srt
[download] 100% of    6.48KiB in 00:00:00 at 28.89KiB/s
Deleting existing file TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-TW.srt
[info] Writing video subtitles to: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-TW.srt


[download] Destination: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.zh-TW.srt
[download] 100% of    6.39KiB in 00:00:00 at 32.02KiB/s
Deleting existing file TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.en.srt
[info] Writing video subtitles to: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.en.srt


[download] Destination: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.en.srt
[download] 100% of    6.61KiB in 00:00:00 at 34.64KiB/s
[info] Downloading video thumbnail 45 ...
[info] Writing video thumbnail 45 to: TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.webp
[download] TED_Ed_Subtitles_SRT\001_破碎的心如何影响你的身体.mp4 has already been downloaded
[download] 100% of   18.24MiB
  ✅ 測試下載成功！
  ✨ 已自動修正大小寫、取消分行並修復標點符號: 001_破碎的心如何影响你的身体.en.srt
  ✨ 已自動修正大小寫、取消分行並修復標點符號: 001_破碎的心如何影响你的身体.zh-CN.srt
  ✨ 已自動修正大小寫、取消分行並修復標點符號: 001_破碎的心如何影响你的身体.zh-Hans.srt
  ✨ 已自動修正大小寫、取消分行並修復標點符號: 001_破碎的心如何影响你的身体.zh-TW.srt

🔍 測試完成！已自動停止，請檢查資料夾中的影片與字幕檔案。
